In [1]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
import urllib.parse
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import seasonal_decompose
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [2]:
USER = "postgres.sqastdkrowpwfactzncc"
PASSWORD = "LABBIOHDS1925"
HOST = "aws-1-ap-northeast-1.pooler.supabase.com"
PORT = "6543"
DBNAME = "postgres"

safe_password = urllib.parse.quote_plus(PASSWORD)
db_url = f"postgresql://{USER}:{safe_password}@{HOST}:{PORT}/{DBNAME}"
engine = create_engine(db_url)

In [3]:
query = """
WITH daily_chem AS (
    SELECT 
        COALESCE(b.booking_date, CURRENT_DATE) AS date,
        SUM(u.value_use) AS chem_used
    FROM public.chemical_usege u
    LEFT JOIN public.booking b ON u.user_id = b.user_id
    GROUP BY b.booking_date
),
daily_booking AS (
    SELECT 
        booking_date AS date, 
        COUNT(booking_id) AS total_users
    FROM public.booking
    GROUP BY booking_date
)
SELECT 
    c.date,
    c.chem_used,
    COALESCE(b.total_users, 0) AS total_users
FROM daily_chem c
LEFT JOIN daily_booking b ON c.date = b.date
ORDER BY c.date ASC;
"""

In [5]:
df_raw = pd.read_sql(query, engine)
if df_raw.empty or len(df_raw) < 10:
    print("⚠️ ข้อมูลจริงใน DB มีน้อยเกินไป ระบบกำลังสร้าง Health Data Mockup สำหรับ Demo...")
    dates = pd.date_range(start='2024-01-01', periods=180, freq='D')
    # จำลองความสัมพันธ์: คนเข้าแล็บเยอะ -> ใช้สารเคมีเยอะ (Staining Process)
    users = [int(abs(np.random.normal(20, 10)) + (10 if d.dayofweek < 5 else 2)) for d in dates]
    chems = [u * np.random.uniform(2.5, 3.5) + np.random.normal(0, 5) for u in users]
    df_raw = pd.DataFrame({'date': dates, 'chem_used': chems, 'total_users': users})

df_raw['date'] = pd.to_datetime(df_raw['date'])
df_raw.set_index('date', inplace=True)

⚠️ ข้อมูลจริงใน DB มีน้อยเกินไป ระบบกำลังสร้าง Health Data Mockup สำหรับ Demo...
